In [1]:
import pandas as pd
path = r"C:\Users\ariel\Desktop\Seminar - Thesis\F-TM-CR\data\accepted_2007_to_2018Q4.csv"

In [2]:
df_head = pd.read_csv(path, nrows=5000, low_memory=False)
df_head.head()
df_head.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Columns: 151 entries, id to settlement_term
dtypes: float64(114), int64(1), object(36)
memory usage: 5.8+ MB


# הסרת שנים + פילטר סטטוס הלוואה

In [3]:

path = r"C:\Users\ariel\Desktop\Seminar - Thesis\F-TM-CR\data\accepted_2007_to_2018Q4.csv"

# טעינת הנתונים - מומלץ להשתמש ב-usecols אם הזיכרון מוגבל, 
# אך כרגע נטען את כל העמודות ונסנן שורות
df = pd.read_csv(path, low_memory=False)

# 1. המרת עמודת התאריך לפורמט datetime
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')

# 2. סינון לפי שנים (2007 עד 2014 כולל)
df = df[(df['issue_d'].dt.year >= 2010) & (df['issue_d'].dt.year <= 2013)]

# 3. הגדרת אוכלוסיית המחקר (יחידת תצפית)
# אנחנו צריכים רק הלוואות עם תוצאה ידועה (Fully Paid או Charged Off/Default)
# הלוואות בסטטוס "Current" אינן רלוונטיות לניבוי דיפולט בדיעבד
target_statuses = ['Fully Paid', 'Charged Off', 'Default']
df = df[df['loan_status'].isin(target_statuses)]

# 4. יצירת משתנה המטרה (Target) - 1 לדיפולט, 0 להחזר תקין
df['is_default'] = df['loan_status'].apply(lambda x: 1 if x in ['Charged Off', 'Default'] else 0)

print(f"מספר תצפיות לאחר סינון שנים וסטטוס: {len(df):,}")

מספר תצפיות לאחר סינון שנים וסטטוס: 221,428


# הסרת עמודות דלף מידע

In [4]:
# רשימת עמודות דלף (Leakage) נפוצות ב-LendingClub
leakage_columns = [
    'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee', 'last_pymnt_d',
    'last_pymnt_amnt', 'last_credit_pull_d', 'settlement_status',
    'settlement_date', 'settlement_amount', 'settlement_term',
    'debt_settlement_flag', 'debt_settlement_flag_date',
    'last_fico_range_high', 'last_fico_range_low',  # ציוני FICO מאותה משיכת אשראי אחרונה כמו last_credit_pull_d -- מידע מהעתיד, דולף בזמן
]

# הסרת העמודות
df = df.drop(columns=[col for col in leakage_columns if col in df.columns])
print(f"מספר עמודות לאחר הסרת דלף מידע: {df.shape[1]}")

מספר עמודות לאחר הסרת דלף מידע: 132


In [5]:
# 1. הסרת עמודות שמעל 50% מהן ריקות (פרט ל-'desc' שאנחנו חייבים)
missing_threshold = 0.5
null_counts = df.isnull().mean()
cols_to_drop = null_counts[null_counts > missing_threshold].index.tolist()

if 'desc' in cols_to_drop:
    cols_to_drop.remove('desc') # שומרים על עמודת הטקסט למרות החסרים

df = df.drop(columns=cols_to_drop)

# 2. הסרת עמודות מזהות שלא תורמות למודל
irrelevant_cols = ['id', 'member_id', 'url']
df = df.drop(columns=[c for c in irrelevant_cols if c in df.columns])

print(f"מספר עמודות סופי: {df.shape[1]}")

מספר עמודות סופי: 78


# יצירת רשימה של עמודות הכרחיות

In [6]:
# 1. הגדרת עמודות הליבה שאסור למחוק
mandatory_columns = [
    'is_default', 'issue_d', 'desc', 'addr_state', # משתני מחקר
    'grade', 'sub_grade', 'annual_inc', 'dti',     # משתני סיכון קלאסיים
    'fico_range_low', 'fico_range_high',           # ציון FICO במועד הנפקת ההלוואה (לא last_fico_* -- זה דולף בזמן, ראה leakage_columns)
    'loan_amnt', 'term', 'int_rate', 'emp_length', 
    'home_ownership', 'verification_status', 'purpose', 'zip_code', 'emp_title', 'title'
]

# 2. זיהוי עמודות טקסט חופשי נוספות (אם קיימות)
# ב-LendingClub יש לעיתים גם את 'emp_title' או 'title', אך 'desc' היא העיקרית
text_columns = ['desc'] 

# 3. יצירת רשימה של עמודות שמועמדות לסינון (כל מה שלא ברשימת החובה)
# הערה: אלגוריתם Boruta הוסר מה-pipeline -- הוא הותאם על כל התקופה במאוחד (2010-2013),
# כך שבחירת המשתנים עצמה דלפה מידע עתידי בזמן. הסינון היחיד שנותר הוא מולטיקולינאריות (בהמשך).
potential_selector_cols = [c for c in df.columns if c not in mandatory_columns]

print(f"עמודות מוגנות: {len(mandatory_columns)}")
print(f"עמודות מועמדות לסינון מולטיקולינאריות: {len(potential_selector_cols)}")

עמודות מוגנות: 20
עמודות מועמדות לסינון מולטיקולינאריות: 58


# בדיקת מוטיקולינאריות

In [7]:
import pandas as pd
import numpy as np

# =================================================================
# STEP 1: DEFINE MANDATORY CORE FEATURES
# =================================================================
mandatory_columns = [
    'is_default', 'issue_d', 'desc', 'addr_state', 
    'grade', 'sub_grade', 'annual_inc', 'dti', 
    'fico_range_low', 'fico_range_high',
    'loan_amnt', 'term', 'int_rate', 'emp_length', 
    'home_ownership', 'verification_status', 'purpose', 
    'zip_code', 'emp_title', 'title'
]

# =================================================================
# STEP 2: FEATURE CONSOLIDATION
# =================================================================
# Boruta was removed from the pipeline: it was fit on the pooled 2010-2013
# data, so the feature selection itself leaked future information across
# time. All remaining numeric columns are candidates; redundancy is handled
# by the multicollinearity filter below.
candidate_features = df.drop(columns=[c for c in mandatory_columns if c in df.columns]) \
                       .select_dtypes(include=[np.number]).columns.tolist()
final_features_list = list(set(candidate_features + mandatory_columns))
df_working = df[final_features_list].copy()

# =================================================================
# STEP 3: MULTICOLLINEARITY ANALYSIS (Threshold > 0.95)
# =================================================================
existing_mandatory = [c for c in mandatory_columns if c in df_working.columns]
features_to_test = df_working.drop(columns=existing_mandatory).select_dtypes(include=[np.number])

# Calculate absolute correlation matrix
corr_matrix = features_to_test.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Extract pairs with correlation > 0.95
high_corr_pairs = []
for column in upper.columns:
    correlated_features = upper.index[upper[column] > 0.95].tolist()
    if correlated_features:
        for corr_feat in correlated_features:
            high_corr_pairs.append({
                'feature_1': column,
                'feature_2': corr_feat,
                'correlation': upper.loc[corr_feat, column]
            })

# =================================================================
# STEP 4: FINAL DATASET CONSTRUCTION & OUTPUT
# =================================================================
if high_corr_pairs:
    pairs_df = pd.DataFrame(high_corr_pairs).sort_values('correlation', ascending=False)
    print(f"\nDetected {len(pairs_df)} pairs with high correlation (> 0.95):")
    print(pairs_df.to_string(index=False)) # Displays the table you wanted
    
    to_drop = pairs_df['feature_2'].unique().tolist()
    df_final_model = df_working.drop(columns=to_drop)
    print(f"\nDropped {len(to_drop)} features due to Multicollinearity.")
else:
    print("\nNo pairs found with correlation > 0.95")
    df_final_model = df_working.copy()

# Final cleaning and verification
if 'desc' in df_final_model.columns:
    df_final_model = df_final_model.drop(columns=['desc'])

print("-" * 30)
print(f"Final feature count: {df_final_model.shape[1]}")
print(f"Is 'zip_code' present? {'YES!' if 'zip_code' in df_final_model.columns else 'No'}")


Detected 6 pairs with high correlation (> 0.95):
          feature_1       feature_2  correlation
           open_acc        num_sats     0.999189
    funded_amnt_inv     funded_amnt     0.998635
num_rev_tl_bal_gt_0 num_actv_rev_tl     0.998491
        tot_cur_bal tot_hi_cred_lim     0.984163
        installment     funded_amnt     0.955495
        installment funded_amnt_inv     0.955109

Dropped 5 features due to Multicollinearity.
------------------------------
Final feature count: 65
Is 'zip_code' present? YES!


# שמירת 2 המסדים

In [8]:
df_final_model.to_csv('lc_basic_database.csv', index=False)

In [9]:
# יצירת מסד שכולל את הפיצ'רים הנבחרים + עמודת התיאור
df_with_text = df[list(df_final_model.columns) + ['desc']]
df_with_text.to_csv('lc_basic_database_+_desc.csv', index=False)
